# Task 1: Exploratory Data Analysis (EDA) - Fraud Risk Platform
## Kaggle Credit Card Transactions Fraud Dataset (Kartik Shenoy / Sparkov)

This notebook conducts a systematic analysis of financial transaction fraud, exploring:
1. Dataset structure and data quality.
2. Class imbalance and fraud prevalence.
3. Amount distributions and outlier analysis.
4. Temporal patterns (hour of day, day of week).
5. Merchant category fraud risk profiling.
6. Customer age and geographical distance patterns.
7. Correlation analysis.
8. Identification of key fraud indicators and data limitations.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

data_path = '../data/raw/fraudTest.csv'
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
df.head()

## 1. Data Quality & Structure Audit
Checking missing values, data types, and duplicate transaction IDs.

In [2]:
print('Missing Values:\n', df.isnull().sum())
print('\nDuplicate Transaction IDs:', df['trans_num'].duplicated().sum())

## 2. Class Imbalance Analysis
Quantifying the rarity of fraudulent transactions.

In [3]:
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100
print(f'Legitimate (0): {fraud_counts[0]:,} ({fraud_pct[0]:.3f}%)')
print(f'Fraudulent (1): {fraud_counts[1]:,} ({fraud_pct[1]:.3f}%)')
print(f'Imbalance Ratio: 1 fraud per ~{int(fraud_counts[0]/fraud_counts[1])} legit transactions')

## 3. Transaction Amount Distribution
Examining spending patterns for legitimate vs fraudulent transactions on log-scale.

In [4]:
print('Legitimate Spend Stats:')
print(df[df['is_fraud'] == 0]['amt'].describe())
print('\nFraudulent Spend Stats:')
print(df[df['is_fraud'] == 1]['amt'].describe())

## 4. Temporal Patterns: Hour of Day
Analyzing when fraud occurs most frequently.

In [5]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['trans_hour'] = df['trans_date_trans_time'].dt.hour
hourly = df.groupby('trans_hour')['is_fraud'].agg(['count', 'sum', 'mean']).reset_index()
hourly.columns = ['hour', 'total', 'fraud_cnt', 'fraud_rate']
hourly['fraud_rate_pct'] = hourly['fraud_rate'] * 100
hourly.sort_values(by='fraud_rate_pct', ascending=False).head()

## 5. Merchant Category Risk
Ranking merchant categories by fraud prevalence.

In [6]:
cat_risk = df.groupby('category')['is_fraud'].agg(['count', 'sum', 'mean']).reset_index()
cat_risk.columns = ['category', 'total', 'fraud_cnt', 'fraud_rate']
cat_risk['fraud_rate_pct'] = cat_risk['fraud_rate'] * 100
cat_risk.sort_values(by='fraud_rate_pct', ascending=False)

## 6. Key Fraud Indicators & Data Limitations

### Potentially Useful Fraud Indicators:
1. **Transaction Hour (Nighttime Surge):** Fraud rate peaks between 22:00 and 03:00 (~1.94% at 22:00 vs ~0.15% at noon).
2. **Merchant Category (Online Shopping / General Net):** `shopping_net` (1.21%) and `misc_net` (0.98%) exhibit over 3x the overall baseline fraud rate.
3. **High Transaction Amount Outliers:** Fraudulent transactions exhibit a median of $371.94 (mean $528.36), compared to legitimate median of $47.15 (mean $67.61).

### Limitations of the Available Data:
- No network/device telemetry (IP address, proxy/VPN flag, browser fingerprint, device ID).
- Absence of real-time multi-factor authentication (3DS challenge status, OTP retries).
- No card velocity windowing directly pre-computed (requires rolling window feature engineering).